# 📡 FM Carrier Frequency Estimation — Compliance Test Framework
> **Sensor:** ANE6 · **Sweep:** `spectrum_ANE6_2026-03-03T19-44-01.csv`  
> **Standards:** ITU-R BS.450-3 · ITU-R SM.328-11 · ETSI EN 302 018-1 · MINTIC Res. 415/2010  
> **SDR:** HackRF One

---
**Pipeline:**  
`CSV` → `parse_psd()` → `detect_carriers()` → `estimate_carrier()` → `report`

**Output fields per carrier:**  
`estimated_freq_hz` · `nominal_freq_hz` · `deviation_hz` · `deviation_khz` · `deviation_ppm`  
`spike_prominence_db` · `spike_width_hz` · `snr_db` · `confidence` · `compliance_status`

## 0 · Environment setup (Deepnote)

In [1]:
# ── Install / upgrade only what is not already present in Deepnote's base image
# Deepnote ships: numpy, pandas, matplotlib, scipy  ✓
# Extra deps used by this notebook:
import importlib, subprocess, sys

REQUIRED = {
    "plotly": "plotly>=5.18",
    "ipywidgets": "ipywidgets>=8.0",
    "tabulate": "tabulate",
}

for module, pkg in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed {pkg}")
    else:
        print(f"OK  {module}")

In [1]:
from __future__ import annotations

import csv
import datetime
import json
import re
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import curve_fit

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tabulate import tabulate

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("All imports OK ✓")

import glob              # File pattern matching (e.g., finding all CSV files)
import os                # Operating system interface (file paths, directory operations)
import time              # Time measurement and delays
from datetime import datetime
import tracemalloc       # Memory allocation tracing (for performance profiling)
try:
    import psutil        # System resource monitoring (CPU, memory usage)
except ImportError:
    psutil = None 

import numpy as np       # Numerical operations, arrays, mathematical functions
import pandas as pd      # DataFrames, data manipulation, CSV handling
import matplotlib.pyplot as plt  # Plotting and visualization
import matplotlib.ticker as ticker  # Customizing axis ticks and labels

import ast               # Safely evaluate string literals (e.g., "[1,2,3]" -> list)
from tqdm import tqdm    # Progress bars for long-running loops

from dataclasses import dataclass  # Create structured data containers (e.g., NoiseFloorResult)
from typing import Optional, Dict
from scipy.ndimage import generic_filter  # Apply sliding window functions to arrays

All imports OK ✓


## 1 · Configuration

In [3]:
# ── File path (Deepnote mounts uploaded files under /work) ──────────────────
CSV_PATH = Path("/work/spectrum_ANE1_2026-03-10T15-31-11.csv")

@dataclass
class FrameworkConfig:
    """All tunable parameters — edit here, not in algorithm cells."""
    # Band limits
    fm_band_min_hz: float = 87.5e6
    fm_band_max_hz: float = 108.0e6
    channel_raster_hz: float = 100e3        # ITU-R 100 kHz FM grid
    # Detection
    min_prominence_db: float = 5.0
    min_peak_distance_hz: float = 100e3
    # Sub-bin interpolation
    interpolation_method: str = "quadratic" # "quadratic" | "gaussian" | "centroid"
    interpolation_half_window: int = 3
    # Noise estimation
    noise_guard_band_hz: float = 200e3
    noise_window_hz: float = 500e3
    # Confidence thresholds
    min_snr_for_full_confidence: float = 20.0
    min_prominence_for_full_confidence: float = 10.0
    # Regulatory — MINTIC Res. 415/2010
    frequency_tolerance_hz: float = 2000.0  # hard limit ±2 kHz
    warning_threshold_hz: float = 1000.0    # soft warning

CFG = FrameworkConfig()
print("Config loaded ✓")
print(f"  Interpolation   : {CFG.interpolation_method}")
print(f"  Tolerance       : ±{CFG.frequency_tolerance_hz/1e3:.1f} kHz")
print(f"  Min prominence  : {CFG.min_prominence_db} dB")

Config loaded ✓
  Interpolation   : quadratic
  Tolerance       : ±2.0 kHz
  Min prominence  : 5.0 dB


## 2 · Data Structures

In [ ]:
@dataclass
class PSDRecord:
    """Parsed PSD sweep from a single CSV file."""
    sensor_id: str
    timestamp: datetime.datetime
    latitude: float
    longitude: float
    freq_hz: np.ndarray
    power_dbm: np.ndarray
    freq_resolution_hz: float
    freq_min_hz: float
    freq_max_hz: float
    power_min_dbm: float
    power_max_dbm: float
    raw_header: dict = field(default_factory=dict)

@dataclass
class CarrierEstimate:
    """All measurands for one detected carrier."""
    # Identification
    sensor_id: str
    timestamp: str
    nominal_freq_hz: Optional[float]
    # Estimation
    estimated_freq_hz: float
    peak_bin_freq_hz: float
    deviation_hz: Optional[float]
    deviation_ppm: Optional[float]
    deviation_khz: Optional[float]
    # Spectral shape
    spike_prominence_db: float
    spike_width_hz: float
    spike_width_bins: float
    peak_power_dbm: float
    snr_db: float
    noise_floor_dbm: float
    # Quality
    confidence: float
    confidence_flags: list
    # Compliance
    compliance_status: str
    tolerance_hz: float
    notes: str

print("Data structures defined ✓")

: 

: 

## 3 · CSV Parser

In [ ]:
_HEADER_PATTERNS = {
    "sensor_id": re.compile(r"^Sensor:\s*(.+)$", re.I),
    "fecha":     re.compile(r"^Fecha:\s*(.+)$", re.I),
    "latitud":   re.compile(r"^Latitud:\s*([\d.eE+\-]+)", re.I),
    "longitud":  re.compile(r"^Longitud:\s*([\d.eE+\-]+)", re.I),
    "n_points":  re.compile(r"^N[uú]mero de puntos:\s*(\d+)", re.I),
    "freq_min":  re.compile(r"^Frecuencia m[íi]nima:\s*([\d.eE+\-]+)\s*Hz", re.I),
    "freq_max":  re.compile(r"^Frecuencia m[aá]xima:\s*([\d.eE+\-]+)\s*Hz", re.I),
    "power_min": re.compile(r"^Potencia m[íi]nima:\s*([\d.eE+\-]+)\s*dBm", re.I),
    "power_max": re.compile(r"^Potencia m[aá]xima:\s*([\d.eE+\-]+)\s*dBm", re.I),
}

_DATE_FORMATS = [
    "%d/%m/%Y, %I:%M:%S %p.",
    "%d/%m/%Y, %I:%M:%S %p",
    "%d/%m/%Y, %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
]

def _parse_timestamp(raw: str) -> datetime.datetime:
    raw = raw.strip().replace("\u202f", " ").replace("\xa0", " ")
    for fmt in _DATE_FORMATS:
        try:
            return datetime.datetime.strptime(raw, fmt)
        except ValueError:
            pass
    try:
        from dateutil import parser as du
        return du.parse(raw)
    except Exception:
        warnings.warn(f"Could not parse timestamp '{raw}', using epoch.")
        return datetime.datetime.utcfromtimestamp(0)

def parse_psd_csv(path: Path) -> PSDRecord:
    raw_header, data_lines, in_data = {}, [], False
    with path.open("r", encoding="utf-8-sig") as fh:
        for line in fh:
            line = line.rstrip("\n\r")
            if in_data:
                data_lines.append(line)
                continue
            if re.match(r"^\s*Frecuencia\s*\(Hz\)", line, re.I):
                in_data = True
                continue
            for key, pat in _HEADER_PATTERNS.items():
                m = pat.search(line)
                if m:
                    raw_header[key] = m.group(1).strip()

    freq_list, power_list = [], []
    for row in data_lines:
        row = row.strip()
        if not row:
            continue
        parts = row.split(",")
        if len(parts) != 2:
            continue
        try:
            freq_list.append(float(parts[0]))
            power_list.append(float(parts[1]))
        except ValueError:
            continue

    freq_hz  = np.array(freq_list,  dtype=np.float64)
    power_dbm = np.array(power_list, dtype=np.float64)

    if len(freq_hz) < 2:
        raise ValueError(f"Insufficient data in {path.name}")

    return PSDRecord(
        sensor_id         = raw_header.get("sensor_id", path.stem),
        timestamp         = _parse_timestamp(raw_header.get("fecha", "")),
        latitude          = float(raw_header.get("latitud", 0.0)),
        longitude         = float(raw_header.get("longitud", 0.0)),
        freq_hz           = freq_hz,
        power_dbm         = power_dbm,
        freq_resolution_hz= float(np.median(np.diff(freq_hz))),
        freq_min_hz       = float(raw_header.get("freq_min", freq_hz.min())),
        freq_max_hz       = float(raw_header.get("freq_max", freq_hz.max())),
        power_min_dbm     = float(raw_header.get("power_min", power_dbm.min())),
        power_max_dbm     = float(raw_header.get("power_max", power_dbm.max())),
        raw_header        = raw_header,
    )

print("Parser defined ✓")

: 

: 

## 4 · Carrier Detection & Interpolation

In [ ]:
def detect_fm_carriers(psd: PSDRecord, cfg: FrameworkConfig) -> list:
    f, p = psd.freq_hz, psd.power_dbm
    mask = (f >= cfg.fm_band_min_hz) & (f <= cfg.fm_band_max_hz)
    if mask.sum() < 10:
        mask = np.ones(len(f), dtype=bool)
        warnings.warn("Sweep outside standard FM band — using full sweep.")
    indices = np.where(mask)[0]
    dist_bins = max(1, int(cfg.min_peak_distance_hz / psd.freq_resolution_hz))
    peaks_sub, _ = find_peaks(p[indices], prominence=cfg.min_prominence_db, distance=dist_bins)
    return indices[peaks_sub].tolist()

# ── Interpolation methods ───────────────────────────────────────────────────
def _quadratic_interp(p, idx, hw):
    lo, hi = max(0, idx-hw), min(len(p)-1, idx+hw)
    x = np.arange(lo, hi+1, dtype=float) - idx
    try:
        c = np.polyfit(x, p[lo:hi+1], 2)
        return float(np.clip(-c[1]/(2*c[0]), -hw, hw)) if c[0] != 0 else 0.0
    except Exception:
        return 0.0

def _gaussian_interp(p, idx, hw):
    lo, hi = max(0, idx-hw), min(len(p)-1, idx+hw)
    x = np.arange(lo, hi+1, dtype=float) - idx
    y = p[lo:hi+1]
    try:
        gauss = lambda x, a, mu, s: a * np.exp(-0.5*((x-mu)/s)**2)
        popt, _ = curve_fit(gauss, x, y, p0=[y.max(), 0.0, hw/2], maxfev=400)
        return float(np.clip(popt[1], -hw, hw))
    except Exception:
        return 0.0

def _centroid_interp(p, idx, hw):
    lo, hi = max(0, idx-hw), min(len(p)-1, idx+hw)
    x = np.arange(lo, hi+1, dtype=float) - idx
    lin = 10**((p[lo:hi+1] - p[idx])/10)
    return float(np.dot(x, lin/lin.sum()))

_INTERP = {"quadratic": _quadratic_interp, "gaussian": _gaussian_interp, "centroid": _centroid_interp}

def interpolate_carrier(psd: PSDRecord, peak_idx: int, cfg: FrameworkConfig) -> float:
    fn = _INTERP.get(cfg.interpolation_method, _quadratic_interp)
    offset = fn(psd.power_dbm, peak_idx, cfg.interpolation_half_window)
    return float(psd.freq_hz[peak_idx] + offset * psd.freq_resolution_hz)

print("Detection + interpolation defined ✓")

: 

: 

## 5 · Spectral Metrics, Confidence & Compliance

In [ ]:
def compute_spectral_metrics(psd: PSDRecord, peak_idx: int, cfg: FrameworkConfig) -> dict:
    p, f, bw = psd.power_dbm, psd.freq_hz, psd.freq_resolution_hz
    n = len(p)
    peak_power = p[peak_idx]

    # Prominence
    peaks_all, props = find_peaks(p, prominence=0)
    prom_db = 0.0
    if peak_idx in peaks_all:
        i = np.where(peaks_all == peak_idx)[0]
        if len(i):
            prom_db = float(props["prominences"][i[0]])
    else:
        hd = int(cfg.min_peak_distance_hz / bw)
        lo, hi = max(0, peak_idx-hd), min(n-1, peak_idx+hd)
        base = min(p[lo:peak_idx].min() if peak_idx > lo else peak_power,
                   p[peak_idx:hi+1].min() if hi > peak_idx else peak_power)
        prom_db = float(peak_power - base)

    # Width at −3 dB
    try:
        widths, *_ = peak_widths(p, [peak_idx], rel_height=0.5)
        w_bins = float(widths[0])
    except Exception:
        w_bins = 0.0

    # Noise floor
    gb = int(cfg.noise_guard_band_hz / bw)
    nw = int(cfg.noise_window_hz / bw)
    noise = np.concatenate([p[max(0, peak_idx-gb-nw):max(0, peak_idx-gb)],
                             p[min(n, peak_idx+gb):min(n, peak_idx+gb+nw)]])
    nf = float(np.median(noise)) if len(noise) else float(np.median(p))

    return {
        "prominence_db": prom_db,
        "width_hz":      w_bins * bw,
        "width_bins":    w_bins,
        "peak_power_dbm": float(peak_power),
        "snr_db":        float(peak_power - nf),
        "noise_floor_dbm": nf,
    }

def compute_confidence(snr_db, prominence_db, width_hz, freq_res, cfg):
    flags = []
    snr_s = min(1.0, max(0.0, snr_db / cfg.min_snr_for_full_confidence))
    if snr_s < 0.5: flags.append(f"LOW_SNR ({snr_db:.1f} dB)")
    prom_s = min(1.0, max(0.0, prominence_db / cfg.min_prominence_for_full_confidence))
    if prom_s < 0.5: flags.append(f"LOW_PROMINENCE ({prominence_db:.1f} dB)")
    sharp_s = max(0.0, 1.0 - max(0, width_hz - 5*freq_res) / (10*freq_res)) if width_hz > 5*freq_res else 1.0
    if sharp_s < 1.0: flags.append(f"WIDE_SPIKE ({width_hz/1e3:.1f} kHz)")
    return round(snr_s*0.45 + prom_s*0.40 + sharp_s*0.15, 4), flags

def nearest_channel(est_hz, cfg):
    ch = round((est_hz - cfg.fm_band_min_hz) / cfg.channel_raster_hz)
    nom = cfg.fm_band_min_hz + ch * cfg.channel_raster_hz
    return float(nom) if abs(est_hz - nom) <= cfg.frequency_tolerance_hz * 5 else None

def assess_compliance(dev_hz, cfg):
    if dev_hz is None: return "UNVERIFIABLE"
    a = abs(dev_hz)
    if a <= cfg.warning_threshold_hz:    return "COMPLIANT"
    elif a <= cfg.frequency_tolerance_hz: return "WARNING"
    else:                                 return "VIOLATION"

print("Metrics, confidence & compliance defined ✓")

: 

: 

## 6 · Estimation Pipeline

In [ ]:
def estimate_carrier(
    psd: PSDRecord, peak_idx: int, cfg: FrameworkConfig,
    frequency_plan: Optional[dict] = None,
) -> CarrierEstimate:

    est_hz   = interpolate_carrier(psd, peak_idx, cfg)
    bin_hz   = float(psd.freq_hz[peak_idx])
    metrics  = compute_spectral_metrics(psd, peak_idx, cfg)

    # Plan lookup
    if frequency_plan:
        best_nom, best_tol = None, None
        for nom, tol in frequency_plan.items():
            if abs(est_hz - nom) < 5*tol:
                if best_nom is None or abs(est_hz-nom) < abs(est_hz-best_nom):
                    best_nom, best_tol = nom, tol
        nominal_hz = best_nom
        tolerance_hz = best_tol or cfg.frequency_tolerance_hz
    else:
        nominal_hz   = nearest_channel(est_hz, cfg)
        tolerance_hz = cfg.frequency_tolerance_hz

    dev_hz  = (est_hz - nominal_hz) if nominal_hz else None
    dev_khz = (dev_hz / 1e3)        if dev_hz is not None else None
    dev_ppm = (dev_hz / nominal_hz * 1e6) if (dev_hz is not None and nominal_hz) else None

    conf, flags = compute_confidence(
        metrics["snr_db"], metrics["prominence_db"],
        metrics["width_hz"], psd.freq_resolution_hz, cfg,
    )

    status = assess_compliance(dev_hz, cfg)
    notes  = " ".join(filter(None, [
        "No matching ITU-R FM channel." if nominal_hz is None else "",
        ("Flags: " + ", ".join(flags)) if flags else "",
        f"Interp: {cfg.interpolation_method}.",
    ]))

    return CarrierEstimate(
        sensor_id          = psd.sensor_id,
        timestamp          = psd.timestamp.isoformat(),
        nominal_freq_hz    = nominal_hz,
        estimated_freq_hz  = round(est_hz, 2),
        peak_bin_freq_hz   = round(bin_hz, 2),
        deviation_hz       = round(dev_hz,  2) if dev_hz  is not None else None,
        deviation_ppm      = round(dev_ppm, 4) if dev_ppm is not None else None,
        deviation_khz      = round(dev_khz, 4) if dev_khz is not None else None,
        spike_prominence_db= round(metrics["prominence_db"],   2),
        spike_width_hz     = round(metrics["width_hz"],        2),
        spike_width_bins   = round(metrics["width_bins"],      2),
        peak_power_dbm     = round(metrics["peak_power_dbm"],  2),
        snr_db             = round(metrics["snr_db"],          2),
        noise_floor_dbm    = round(metrics["noise_floor_dbm"], 2),
        confidence         = conf,
        confidence_flags   = flags,
        compliance_status  = status,
        tolerance_hz       = tolerance_hz,
        notes              = notes,
    )

def run_estimation(csv_path, cfg=None, frequency_plan=None):
    cfg = cfg or FrameworkConfig()
    psd = parse_psd_csv(Path(csv_path))
    peaks = detect_fm_carriers(psd, cfg)
    estimates = [estimate_carrier(psd, idx, cfg, frequency_plan) for idx in peaks]
    return psd, estimates

print("Pipeline defined ✓")

: 

: 

## 7 · Use Case — ANE6 Sweep

In [ ]:
# ── Run ─────────────────────────────────────────────────────────────────────
psd, estimates = run_estimation(CSV_PATH, cfg=CFG)

# ── Sweep summary ───────────────────────────────────────────────────────────
print(f"Sensor        : {psd.sensor_id}")
print(f"Timestamp     : {psd.timestamp.isoformat()}")
print(f"Location      : {psd.latitude:.6f}°N  {psd.longitude:.6f}°E")
print(f"Sweep range   : {psd.freq_min_hz/1e6:.3f} – {psd.freq_max_hz/1e6:.3f} MHz")
print(f"Resolution    : {psd.freq_resolution_hz:.1f} Hz")
print(f"Bins          : {len(psd.freq_hz):,}")
print(f"Carriers found: {len(estimates)}")

: 

: 

In [ ]:
# ── Results table ────────────────────────────────────────────────────────────
rows = []
for e in estimates:
    status_icon = {"COMPLIANT": "✅", "WARNING": "⚠️", "VIOLATION": "🚨", "UNVERIFIABLE": "❓"}.get(e.compliance_status, "?")
    rows.append([
        f"{e.nominal_freq_hz/1e6:.4f}" if e.nominal_freq_hz else "N/A",
        f"{e.estimated_freq_hz/1e6:.6f}",
        f"{e.deviation_hz:+.1f}"  if e.deviation_hz  is not None else "N/A",
        f"{e.deviation_khz:+.3f}" if e.deviation_khz is not None else "N/A",
        f"{e.spike_prominence_db:.2f}",
        f"{e.spike_width_hz:.1f}",
        f"{e.snr_db:.2f}",
        f"{e.confidence:.3f}",
        f"{status_icon} {e.compliance_status}",
    ])

headers = ["Nominal (MHz)", "Estimated (MHz)", "Dev (Hz)", "Dev (kHz)",
           "Prominence (dB)", "Width (Hz)", "SNR (dB)", "Confidence", "Status"]

print(tabulate(rows, headers=headers, tablefmt="github"))

: 

: 

## 8 · Visualizations

In [ ]:
# ── 8.1  Full PSD with annotated carriers ────────────────────────────────────
COLOR_MAP = {"COMPLIANT": "#2ecc71", "WARNING": "#f39c12", "VIOLATION": "#e74c3c", "UNVERIFIABLE": "#95a5a6"}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=psd.freq_hz / 1e6, y=psd.power_dbm,
    mode="lines", name="PSD",
    line=dict(color="#3498db", width=0.8),
    hovertemplate="%{x:.4f} MHz<br>%{y:.2f} dBm<extra></extra>",
))

for e in estimates:
    col = COLOR_MAP.get(e.compliance_status, "gray")
    label = f"{e.estimated_freq_hz/1e6:.4f} MHz"
    fig.add_vline(x=e.estimated_freq_hz/1e6, line=dict(color=col, dash="dot", width=1.5))
    fig.add_annotation(
        x=e.estimated_freq_hz/1e6, y=e.peak_power_dbm + 1.5,
        text=label, showarrow=True, arrowhead=2, arrowcolor=col,
        font=dict(size=10, color=col), bgcolor="rgba(0,0,0,0.6)",
        bordercolor=col, borderwidth=1,
    )

fig.update_layout(
    title=dict(text=f"Power Spectral Density — {psd.sensor_id}  ({psd.timestamp.strftime('%Y-%m-%d %H:%M:%S')})",
               font=dict(size=14)),
    xaxis_title="Frequency (MHz)",
    yaxis_title="Power (dBm)",
    template="plotly_dark",
    height=450,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.06),
    margin=dict(l=60, r=20, t=70, b=50),
)
fig.show()

: 

: 

In [ ]:
# ── 8.2  Per-carrier zoom + metrics dashboard ────────────────────────────────
if estimates:
    ZOOM_HZ = 500e3  # ±500 kHz window
    n_est = len(estimates)
    cols = min(n_est, 3)
    rows_n = (n_est + cols - 1) // cols

    fig2 = make_subplots(
        rows=rows_n, cols=cols,
        subplot_titles=[f"{e.estimated_freq_hz/1e6:.4f} MHz" for e in estimates],
        vertical_spacing=0.15, horizontal_spacing=0.08,
    )

    for i, e in enumerate(estimates):
        r, c = divmod(i, cols)
        mask = np.abs(psd.freq_hz - e.estimated_freq_hz) <= ZOOM_HZ
        xz = psd.freq_hz[mask] / 1e6
        yz = psd.power_dbm[mask]
        col_c = COLOR_MAP.get(e.compliance_status, "gray")

        fig2.add_trace(
            go.Scatter(x=xz, y=yz, mode="lines", name=f"{e.estimated_freq_hz/1e6:.4f}",
                       line=dict(color="#7fb3d3", width=1), showlegend=False,
                       hovertemplate="%{x:.5f} MHz<br>%{y:.2f} dBm<extra></extra>"),
            row=r+1, col=c+1,
        )
        # Carrier line
        fig2.add_vline(x=e.estimated_freq_hz/1e6, line=dict(color=col_c, dash="dash", width=1.5),
                       row=r+1, col=c+1)
        # Noise floor
        fig2.add_hline(y=e.noise_floor_dbm, line=dict(color="#f0e68c", dash="dot", width=1),
                       row=r+1, col=c+1)
        # Annotation
        dev_str = f"Δ = {e.deviation_hz:+.0f} Hz" if e.deviation_hz is not None else "Unplanned"
        fig2.add_annotation(
            x=e.estimated_freq_hz/1e6, y=e.peak_power_dbm,
            text=f"{dev_str}<br>SNR {e.snr_db:.1f} dB<br>Conf {e.confidence:.2f}",
            showarrow=True, arrowhead=1, arrowcolor=col_c,
            bgcolor="rgba(0,0,0,0.7)", font=dict(size=9, color=col_c),
            bordercolor=col_c, row=r+1, col=c+1,
        )

    fig2.update_layout(
        title="Carrier Detail View — ±500 kHz window",
        template="plotly_dark", height=350 * rows_n,
        margin=dict(l=50, r=20, t=80, b=40),
    )
    fig2.show()

: 

: 

In [ ]:
# ── 8.3  Quality metrics bar chart ───────────────────────────────────────────
if estimates:
    labels = [f"{e.estimated_freq_hz/1e6:.4f} MHz" for e in estimates]
    fig3 = make_subplots(
        rows=1, cols=3,
        subplot_titles=["SNR (dB)", "Prominence (dB)", "Confidence"],
    )
    colors = [COLOR_MAP.get(e.compliance_status, "gray") for e in estimates]

    for col_idx, (vals, name) in enumerate([
        ([e.snr_db           for e in estimates], "SNR"),
        ([e.spike_prominence_db for e in estimates], "Prominence"),
        ([e.confidence       for e in estimates], "Confidence"),
    ], start=1):
        fig3.add_trace(
            go.Bar(x=labels, y=vals, marker_color=colors,
                   name=name, showlegend=False),
            row=1, col=col_idx,
        )

    fig3.update_layout(
        title="Quality Metrics per Carrier",
        template="plotly_dark", height=320,
        margin=dict(l=50, r=20, t=60, b=80),
    )
    fig3.show()

: 

: 

## 9 · Export

In [ ]:
_CSV_FIELDS = [
    "sensor_id", "timestamp", "nominal_freq_hz", "estimated_freq_hz",
    "peak_bin_freq_hz", "deviation_hz", "deviation_khz", "deviation_ppm",
    "spike_prominence_db", "spike_width_hz", "spike_width_bins",
    "peak_power_dbm", "snr_db", "noise_floor_dbm",
    "confidence", "compliance_status", "tolerance_hz", "notes",
]

def export_csv(estimates, path):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=_CSV_FIELDS)
        w.writeheader()
        for e in estimates:
            row = asdict(e)
            row.pop("confidence_flags", None)
            w.writerow({k: row.get(k, "") for k in _CSV_FIELDS})
    print(f"CSV  → {path}")

def export_json(psd, estimates, path):
    payload = {
        "metadata": {
            "sensor_id": psd.sensor_id,
            "timestamp": psd.timestamp.isoformat(),
            "latitude":  psd.latitude,
            "longitude": psd.longitude,
            "freq_min_hz": psd.freq_min_hz,
            "freq_max_hz": psd.freq_max_hz,
            "freq_resolution_hz": psd.freq_resolution_hz,
            "n_points": len(psd.freq_hz),
            "framework_version": "1.0.0",
        },
        "carriers": [asdict(e) for e in estimates],
    }
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    print(f"JSON → {path}")

# ── Export to /work (Deepnote persistent storage) ────────────────────────────
stem = CSV_PATH.stem
export_csv(estimates,  f"/work/{stem}_report.csv")
export_json(psd, estimates, f"/work/{stem}_report.json")

# ── Compliance summary ────────────────────────────────────────────────────────
statuses = [e.compliance_status for e in estimates]
print(f"\nSummary  |  Detected: {len(estimates)}  |  "
      f"Compliant: {statuses.count('COMPLIANT')}  |  "
      f"Warning: {statuses.count('WARNING')}  |  "
      f"Violation: {statuses.count('VIOLATION')}  |  "
      f"Unverifiable: {statuses.count('UNVERIFIABLE')}")

: 

: 

# 10\.  Multiple entries

In [ ]:
def load_node_csvs(
    root_pattern: str,
    *,
    exclude_basenames=None,
    read_csv_kwargs=None,
    show_progress: bool = True,
    sort_files: bool = False,
):
    """Load CSV files matching pattern into dictionary of DataFrames."""
    exclude = {
        os.path.splitext(os.path.basename(x))[0]
        for x in (exclude_basenames or [])
    }
    read_csv_kwargs = read_csv_kwargs or {}

    archivos_all = glob.glob(root_pattern)
    if sort_files:
        archivos_all.sort()

    # Generator instead of full list
    archivos_iter = (
        (path, os.path.splitext(os.path.basename(path))[0])
        for path in archivos_all
        if os.path.splitext(os.path.basename(path))[0] not in exclude
    )

    iterator = tqdm(archivos_iter, desc="Loading CSVs") if show_progress else archivos_iter

    datos_nodos = {}
    errors = []

    try:
        for archivo, nombre_nodo in iterator:
            try:
                datos_nodos[nombre_nodo] = pd.read_csv(archivo, **read_csv_kwargs)
            except (pd.errors.ParserError, FileNotFoundError, OSError) as e:
                error_msg = f"{type(e).__name__}: {e}"
                errors.append((archivo, error_msg))
                if show_progress:
                    tqdm.write(f"   {archivo}: {error_msg}")
    finally:
        if show_progress and hasattr(iterator, 'close'):
            iterator.close()

    archivos_cargados = list(datos_nodos.keys())
    return datos_nodos, archivos_cargados, errors


def parse_pxx_cell(pxx_raw):
    """Parse PSD data into 1D float array with comprehensive error handling."""
    
    # 1. NumPy fast path
    if isinstance(pxx_raw, np.ndarray):
        return pxx_raw.astype(float, copy=False).ravel()
    
    # 2. Sequence fast path  
    if isinstance(pxx_raw, (list, tuple)):
        return np.array(pxx_raw, dtype=float).ravel()
    
    # 3. Pandas Series
    if isinstance(pxx_raw, pd.Series):
        return pxx_raw.to_numpy(dtype=float).ravel()
    
    # 4. String parsing with fallbacks
    s = str(pxx_raw).strip()
    if not s:
        return np.array([], dtype=float)
    
    # Try bracket notation
    if s.startswith("[") and s.endswith("]"):
        body = s[1:-1].replace(",", " ").replace("nan", "NaN").replace("inf", "Inf")
        try:
            parts = body.split()
            return np.fromiter(map(float, parts), dtype=float, count=len(parts))
        except ValueError:
            pass
    
    # Safe fallback
    try:
        return np.asarray(ast.literal_eval(s), dtype=float).ravel()
    except Exception as e:
        raise ValueError(f"Cannot parse '{s[:30]}...': {e}") from e

: 

: 

In [ ]:
# =============================================================================
# 1. CONFIGURATION & PARAMETERS
# =============================================================================

ROOT_PATTERN = "DatasetFM-no-gains-88M-108M/Node*.csv"


EXCLUDE_BASENAME  = {
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota", # not enough data
    #"DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node5-Bogota", # atipical 
    #"DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node6-Bogota", # poor quality -- brocken mux 
    #"DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota", # poorest measure quality -- without antenna
}

datos_nodos, archivos, errors = load_node_csvs(
    ROOT_PATTERN,
    exclude_basenames=EXCLUDE_BASENAME,
    show_progress=True,
)

print(f"Loaded {len(datos_nodos)} nodes from {len(archivos)} files.")
if errors:
    print("Errors:")
    for f, msg in errors:
        print(f"  {f}: {msg}")

## ── Infer true FFT size from the first valid row ────────────────────────────

ROW_PLOT = 0
global_min = 103

_first_valid = next(
    (df for df in datos_nodos.values() if "pxx" in df.columns and len(df) > ROW_PLOT),
    None
)
if _first_valid is None:
    raise RuntimeError("No valid node found to infer FFT size.")

# Node organization (sorted for legend consistency)
NODE_NAMES  = sorted(datos_nodos.keys(), key=lambda x: x.split('/')[-1])
N_NODES     = len(NODE_NAMES)
NODE_LABELS = [n.split('/')[-1] for n in NODE_NAMES]

# PSD histogram binning strategy
pxx_len = parse_pxx_cell(_first_valid["pxx"].iloc[ROW_PLOT]).size  # Expected PSD array length

# Calculate N_BINS using average of Sturges-like and sqrt heuristics
N_BINS = int(np.ceil((np.log2(pxx_len) + np.sqrt(pxx_len)) / 2))

# Global value range for consistent histogram comparison across nodes
# Assumption: PSD values are in dB scale 
GLOBAL_RANGE = (-27, 6)  # (min_dB, max_dB)

# Pre-allocate shared bin edges so every node uses the same grid
bin_edges   = np.linspace(GLOBAL_RANGE[0], GLOBAL_RANGE[1], N_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  # For plotting

pxx_node = {}

for nombre_nodo, df in datos_nodos.items():
    if "pxx" not in df.columns:
        print(f" {nombre_nodo}: missing 'pxx' column")
        continue

    pxx = None
    for row in (0, global_min):
        try:
            candidate = parse_pxx_cell(df["pxx"].iloc[row])
            if candidate.size > 0:
                pxx = candidate
                break  # stop at first valid parse
        except Exception as e:
            print(f" {nombre_nodo}: failed to parse pxx at row {row} ({type(e).__name__}: {e})")

    if pxx is None or pxx.size == 0:
        print(f" {nombre_nodo}: empty pxx")
        continue

    # Placeholder RF axis 
    freqs_mhz = np.linspace(88.0, 108.0, pxx.size, endpoint=True)

    # Store both PSD and frequency axis for clarity
    pxx_node[nombre_nodo] = {
    "freqs_mhz": freqs_mhz,
    "pxx": pxx
}

: 

: 

In [ ]:
# =============================================================================
# FM CARRIER DETECTION — BATCH SUMMARY OVER ALL NODES
# =============================================================================
# CSV format (one row = one sweep):
#   id, mac, campaign_id, pxx (JSON array str), start_freq_hz, end_freq_hz,
#   timestamp (ms epoch), lat, lng, excursion_*, depth_*, created_at
#
# Pipeline per node:
#   load CSV  →  parse all pxx rows  →  average in linear scale  →  detect
#   →  estimate (sub-bin interp + metrics)  →  append to summary_df
#
# Output DataFrames:
#   summary_df  — one row per detected carrier  (all nodes)
#   carrier_df  — per-channel cross-node stats
#   node_df     — per-node aggregate
# =============================================================================

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths
from dataclasses import dataclass
from typing import Optional
import warnings, glob, ast
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ─── Configuration ────────────────────────────────────────────────────────────

@dataclass
class DetectionConfig:
    fm_band_min_mhz: float     = 88.0
    fm_band_max_mhz: float     = 108.0
    channel_raster_mhz: float  = 0.1        # 100 kHz ITU-R grid
    average_in_linear: bool    = True        # True=avg linear→dB; False=avg dB
    min_prominence_db: float   = 5.0
    min_peak_distance_mhz: float = 0.1
    interp_half_window: int    = 3
    noise_guard_mhz: float     = 0.2
    noise_window_mhz: float    = 0.5
    min_snr_full: float        = 20.0
    min_prom_full: float       = 10.0
    tolerance_khz: float       = 2.0        # MINTIC Res. 415/2010
    warning_khz: float         = 1.0

CFG = DetectionConfig()


# ─── PXX cell parser ─────────────────────────────────────────────────────────

def parse_pxx_cell(cell: str) -> np.ndarray:
    try:
        return np.array(ast.literal_eval(cell), dtype=np.float64)
    except Exception:
        return np.fromstring(str(cell).strip("[]"), sep=",", dtype=np.float64)


# ─── Node loader: CSV → temporally averaged PSD ───────────────────────────────

def load_node_psd(csv_path, cfg: DetectionConfig = CFG) -> Optional[dict]:
    path = Path(csv_path)
    try:
        df = pd.read_csv(path)
    except Exception:
        return None

    if "pxx" not in df.columns or df.empty:
        return None

    arrays = []
    for cell in df["pxx"]:
        try:
            arr = parse_pxx_cell(str(cell))
            if arr.size > 0:
                arrays.append(arr)
        except Exception:
            continue

    if not arrays:
        return None

    lengths  = np.array([a.size for a in arrays])
    mode_len = int(pd.Series(lengths).mode().iloc[0])
    arrays   = [a for a in arrays if a.size == mode_len]
    stack    = np.vstack(arrays)                           # (n_sweeps, n_bins)

    if cfg.average_in_linear:
        pxx_avg = 10 * np.log10(np.maximum(np.mean(10 ** (stack / 10), axis=0), 1e-30))
    else:
        pxx_avg = np.mean(stack, axis=0)

    row0      = df.dropna(subset=["start_freq_hz", "end_freq_hz"]).iloc[0]
    freqs_mhz = np.linspace(float(row0["start_freq_hz"]) / 1e6,
                             float(row0["end_freq_hz"])   / 1e6,
                             mode_len, endpoint=True)
    ts = df["timestamp"].dropna()

    return {
        "node_name":       path.stem,
        "freqs_mhz":       freqs_mhz,
        "pxx":             pxx_avg,
        "n_sweeps":        len(arrays),
        "timestamp_first": int(ts.min()) if not ts.empty else None,
        "timestamp_last":  int(ts.max()) if not ts.empty else None,
        "lat": float(df["lat"].dropna().iloc[0]) if "lat" in df and df["lat"].notna().any() else None,
        "lng": float(df["lng"].dropna().iloc[0]) if "lng" in df and df["lng"].notna().any() else None,
    }


# ─── Spectral helpers ─────────────────────────────────────────────────────────

def _quadratic_offset(p, idx, hw):
    lo, hi = max(0, idx - hw), min(len(p) - 1, idx + hw)
    x = np.arange(lo, hi + 1, dtype=float) - idx
    try:
        c = np.polyfit(x, p[lo:hi + 1], 2)
        return float(np.clip(-c[1] / (2 * c[0]), -hw, hw)) if c[0] != 0 else 0.0
    except Exception:
        return 0.0

def _noise_floor(p, idx, df_mhz, cfg):
    gb, nw, n = max(1, int(cfg.noise_guard_mhz / df_mhz)), max(1, int(cfg.noise_window_mhz / df_mhz)), len(p)
    samples = np.concatenate([p[max(0, idx-gb-nw):max(0, idx-gb)],
                               p[min(n, idx+gb):min(n, idx+gb+nw)]])
    return float(np.median(samples)) if samples.size > 0 else float(np.median(p))

def _prominence(p, idx, df_mhz, cfg):
    peaks_all, props = find_peaks(p, prominence=0)
    where = np.where(peaks_all == idx)[0]
    if where.size:
        return float(props["prominences"][where[0]])
    hd = max(1, int(cfg.min_peak_distance_mhz / df_mhz))
    lo, hi = max(0, idx - hd), min(len(p) - 1, idx + hd)
    base = min(p[lo:idx].min() if idx > lo else p[idx],
               p[idx:hi+1].min() if hi > idx else p[idx])
    return float(p[idx] - base)

def _confidence(snr_db, prom_db, width_bins, cfg):
    flags  = []
    s_snr  = min(1.0, max(0.0, snr_db  / cfg.min_snr_full))
    s_prom = min(1.0, max(0.0, prom_db / cfg.min_prom_full))
    if s_snr  < 0.5: flags.append(f"LOW_SNR({snr_db:.1f}dB)")
    if s_prom < 0.5: flags.append(f"LOW_PROM({prom_db:.1f}dB)")
    s_sharp = max(0.0, 1.0 - max(0.0, width_bins - 5) / 10) if width_bins > 5 else 1.0
    if s_sharp < 1.0: flags.append(f"WIDE({width_bins:.1f}bins)")
    return round(s_snr * 0.45 + s_prom * 0.40 + s_sharp * 0.15, 4), flags

def _nearest_channel(est_mhz, cfg):
    ch  = round((est_mhz - cfg.fm_band_min_mhz) / cfg.channel_raster_mhz)
    nom = cfg.fm_band_min_mhz + ch * cfg.channel_raster_mhz
    return round(nom, 1) if abs(est_mhz - nom) <= cfg.tolerance_khz / 1e3 * 5 else None


# ─── Estimate one carrier ─────────────────────────────────────────────────────

def _estimate_one(node_name, freqs_mhz, p, peak_idx, cfg):
    df_mhz  = float(np.median(np.diff(freqs_mhz)))
    est_mhz = freqs_mhz[peak_idx] + _quadratic_offset(p, peak_idx, cfg.interp_half_window) * df_mhz

    try:
        w_bins = float(peak_widths(p, [peak_idx], rel_height=0.5)[0][0])
    except Exception:
        w_bins = 0.0

    prom_db = _prominence(p, peak_idx, df_mhz, cfg)
    nf_dbm  = _noise_floor(p, peak_idx, df_mhz, cfg)
    snr_db  = float(p[peak_idx] - nf_dbm)
    conf, flags = _confidence(snr_db, prom_db, w_bins, cfg)

    nom_mhz = _nearest_channel(est_mhz, cfg)
    dev_hz  = (est_mhz - nom_mhz) * 1e6 if nom_mhz else None
    dev_khz = dev_hz / 1e3               if dev_hz is not None else None
    dev_ppm = dev_hz / (nom_mhz * 1e6) * 1e6 if (dev_hz is not None and nom_mhz) else None

    if   dev_hz is None:                        status = "UNVERIFIABLE"
    elif abs(dev_hz) <= cfg.warning_khz   * 1e3: status = "COMPLIANT"
    elif abs(dev_hz) <= cfg.tolerance_khz * 1e3: status = "WARNING"
    else:                                        status = "VIOLATION"

    return {
        "node":                node_name,
        "nominal_mhz":         nom_mhz,
        "estimated_mhz":       round(est_mhz, 6),
        "peak_bin_mhz":        round(float(freqs_mhz[peak_idx]), 6),
        "deviation_hz":        round(dev_hz,  2) if dev_hz  is not None else None,
        "deviation_khz":       round(dev_khz, 4) if dev_khz is not None else None,
        "deviation_ppm":       round(dev_ppm, 4) if dev_ppm is not None else None,
        "spike_prominence_db": round(prom_db, 2),
        "spike_width_hz":      round(w_bins * df_mhz * 1e6, 1),
        "spike_width_bins":    round(w_bins, 2),
        "peak_power_dbm":      round(float(p[peak_idx]), 2),
        "snr_db":              round(snr_db,  2),
        "noise_floor_dbm":     round(nf_dbm,  2),
        "confidence":          conf,
        "confidence_flags":    "|".join(flags) if flags else "",
        "compliance_status":   status,
    }


# ─── Batch runner ─────────────────────────────────────────────────────────────

def run_batch_detection(pxx_node: dict, cfg=CFG, verbose=True) -> pd.DataFrame:
    records, skipped = [], []
    for node_name in sorted(pxx_node.keys()):
        entry = pxx_node[node_name]
        freqs = np.asarray(entry["freqs_mhz"], dtype=np.float64)
        p     = np.asarray(entry["pxx"],        dtype=np.float64)
        if p.size < 10:
            skipped.append(node_name); continue

        df_mhz    = float(np.median(np.diff(freqs)))
        dist_bins = max(1, int(cfg.min_peak_distance_mhz / df_mhz))
        peaks, _  = find_peaks(p, prominence=cfg.min_prominence_db, distance=dist_bins)

        if verbose:
            n_sw  = entry.get("n_sweeps", "?")
            label = node_name.split("/")[-1]
            print(f"  {label:<28}  sweeps={str(n_sw):>3}  carriers={len(peaks):>2}")

        for idx in peaks:
            records.append(_estimate_one(node_name, freqs, p, int(idx), cfg))

    if skipped and verbose:
        print(f"\n  Skipped (empty): {[s.split('/')[-1] for s in skipped]}")

    if not records:
        print("No carriers detected."); return pd.DataFrame()

    col_order = ["node","nominal_mhz","estimated_mhz","peak_bin_mhz",
                 "deviation_hz","deviation_khz","deviation_ppm",
                 "spike_prominence_db","spike_width_hz","spike_width_bins",
                 "peak_power_dbm","snr_db","noise_floor_dbm",
                 "confidence","confidence_flags","compliance_status"]
    df = pd.DataFrame(records)
    return df[[c for c in col_order if c in df.columns]]


# ─── Loaders from glob pattern ────────────────────────────────────────────────

def build_pxx_node_averaged(pattern, exclude_basenames=None, cfg=CFG, verbose=True) -> dict:
    exclude_basenames = exclude_basenames or set()
    result, errors = {}, []
    for fpath in sorted(glob.glob(pattern, recursive=True)):
        stem = str(Path(fpath).with_suffix(""))
        if any(ex in stem for ex in exclude_basenames):
            if verbose: print(f"  Excluded: {Path(fpath).stem}")
            continue
        entry = load_node_psd(fpath, cfg)
        if entry is None:
            errors.append(fpath); continue
        result[entry["node_name"]] = entry
        if verbose:
            print(f"  Loaded  {entry['node_name']:<28}  "
                  f"sweeps={entry['n_sweeps']:>3}  bins={entry['pxx'].size:>6}")
    if errors and verbose:
        print(f"\n  Failed: {errors}")
    return result


# ─── Summary tables ───────────────────────────────────────────────────────────

def carrier_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    status_rank = {"VIOLATION":3,"WARNING":2,"COMPLIANT":1,"UNVERIFIABLE":0}
    out = df.groupby("nominal_mhz", dropna=False).agg(
        n_nodes         =("node",               "nunique"),
        dev_hz_mean     =("deviation_hz",       "mean"),
        dev_hz_std      =("deviation_hz",       "std"),
        dev_hz_max_abs  =("deviation_hz",       lambda x: x.abs().max()),
        snr_mean_db     =("snr_db",             "mean"),
        snr_min_db      =("snr_db",             "min"),
        prominence_mean =("spike_prominence_db","mean"),
        confidence_mean =("confidence",         "mean"),
        confidence_min  =("confidence",         "min"),
    ).reset_index()
    worst = (df.assign(_r=df["compliance_status"].map(status_rank))
               .groupby("nominal_mhz", dropna=False)
               .apply(lambda g: g.loc[g["_r"].idxmax(), "compliance_status"])
               .reset_index(name="worst_status"))
    return out.merge(worst, on="nominal_mhz", how="left").sort_values("nominal_mhz").round(3)


def node_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    counts = df.groupby(["node","compliance_status"]).size().unstack(fill_value=0)
    for col in ["COMPLIANT","WARNING","VIOLATION","UNVERIFIABLE"]:
        if col not in counts.columns: counts[col] = 0
    agg = df.groupby("node").agg(
        n_carriers      =("estimated_mhz",  "count"),
        snr_mean_db     =("snr_db",         "mean"),
        snr_min_db      =("snr_db",         "min"),
        confidence_mean =("confidence",     "mean"),
        confidence_min  =("confidence",     "min"),
        dev_hz_mean_abs =("deviation_hz",   lambda x: x.abs().mean()),
        dev_hz_max_abs  =("deviation_hz",   lambda x: x.abs().max()),
    ).reset_index().round(3)
    out = agg.merge(counts[["COMPLIANT","WARNING","VIOLATION","UNVERIFIABLE"]].reset_index(),
                    on="node", how="left")
    out["node_label"] = out["node"].str.split("/").str[-1]
    cols = ["node_label","n_carriers","COMPLIANT","WARNING","VIOLATION","UNVERIFIABLE",
            "snr_mean_db","snr_min_db","confidence_mean","confidence_min",
            "dev_hz_mean_abs","dev_hz_max_abs","node"]
    return out[[c for c in cols if c in out.columns]].sort_values("node_label")


def print_summary(df: pd.DataFrame, cfg=CFG) -> None:
    sep = "═" * 92
    print(f"\n{sep}")
    print(f"  FM CARRIER ESTIMATION — BATCH SUMMARY")
    print(f"  Nodes: {df['node'].nunique()}  |  Detections: {len(df)}  |  "
          f"Tolerance: ±{cfg.tolerance_khz} kHz  |  "
          f"Avg: {'linear' if cfg.average_in_linear else 'dB-domain'}")
    print(sep)

    vc = df["compliance_status"].value_counts()
    print(f"\n  COMPLIANCE BREAKDOWN")
    icon_map = {"COMPLIANT":"✅","WARNING":"⚠️ ","VIOLATION":"🚨","UNVERIFIABLE":"❓"}
    for s in ["COMPLIANT","WARNING","VIOLATION","UNVERIFIABLE"]:
        n = vc.get(s, 0)
        print(f"    {icon_map[s]} {s:<14} {n:>3}  {'█' * n}")

    cs = carrier_summary(df)
    if not cs.empty:
        print(f"\n  PER-CHANNEL CROSS-NODE STATISTICS")
        print(f"  {'Chan':>8} {'#N':>4} {'Dev̄(Hz)':>10} {'σ(Hz)':>8} "
              f"{'|D|max':>8} {'SNR̄':>7} {'Conf̄':>7} {'Worst':>14}")
        print("  " + "─" * 72)
        for _, r in cs.iterrows():
            ch  = f"{r['nominal_mhz']:.1f}" if pd.notna(r["nominal_mhz"]) else "N/A"
            dm  = f"{r['dev_hz_mean']:+.1f}" if pd.notna(r["dev_hz_mean"]) else "N/A"
            ds  = f"{r['dev_hz_std']:.1f}"   if pd.notna(r["dev_hz_std"])  else "—"
            dma = f"{r['dev_hz_max_abs']:.1f}" if pd.notna(r["dev_hz_max_abs"]) else "N/A"
            ws  = r.get("worst_status","")
            print(f"  {ch:>8} {int(r['n_nodes']):>4} {dm:>10} {ds:>8} {dma:>8} "
                  f"{r['snr_mean_db']:>7.1f} {r['confidence_mean']:>7.3f} "
                  f"  {icon_map.get(ws,'')}{ws:>12}")

    ns = node_summary(df)
    if not ns.empty:
        print(f"\n  PER-NODE SUMMARY")
        print(f"  {'Node':<28} {'Cars':>4} {'OK':>4} {'WRN':>4} {'VIO':>4} "
              f"{'SNR̄':>7} {'Cmin':>7} {'|D|̄Hz':>9} {'|D|max':>9}")
        print("  " + "─" * 84)
        for _, r in ns.iterrows():
            print(f"  {r['node_label']:<28} {int(r['n_carriers']):>4} "
                  f"{int(r['COMPLIANT']):>4} {int(r['WARNING']):>4} {int(r['VIOLATION']):>4} "
                  f"{r['snr_mean_db']:>7.1f} {r['confidence_min']:>7.3f} "
                  f"{r['dev_hz_mean_abs']:>9.1f} {r['dev_hz_max_abs']:>9.1f}")
    print(f"\n{sep}\n")


# =============================================================================
# EXECUTION  —  choose one block
# =============================================================================

# ── Block 1 (default): pxx_node already exists from your loader cell ─────────
#    Adds n_sweeps=1 placeholder so the logger doesn't crash if missing.

for v in pxx_node.values():
    v.setdefault("n_sweeps", 1)

print("Running batch carrier detection...")
print(f"  Nodes: {len(pxx_node)}  |  averaging: upstream (your loader)\n")
summary_df = run_batch_detection(pxx_node, cfg=CFG, verbose=True)

# ── Block 2: load & average everything here (uncomment to use) ────────────────
# ROOT_PATTERN      = "DatasetFM-no-gains-88M-108M/Node*.csv"
# EXCLUDE_BASENAME  = {"DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota"}
# print("Loading & averaging node CSVs...")
# pxx_node = build_pxx_node_averaged(ROOT_PATTERN, EXCLUDE_BASENAME, CFG)
# summary_df = run_batch_detection(pxx_node, cfg=CFG, verbose=True)

# ── Report + derived DataFrames ───────────────────────────────────────────────
print_summary(summary_df, cfg=CFG)

carrier_df  = carrier_summary(summary_df)
node_df     = node_summary(summary_df)
violations  = summary_df[summary_df["compliance_status"] == "VIOLATION"]
warnings_df = summary_df[summary_df["compliance_status"] == "WARNING"]
low_conf    = summary_df[summary_df["confidence"] < 0.5]

print(f"DataFrames ready:")
print(f"  summary_df   → {len(summary_df):>4} rows  (one per detected carrier)")
print(f"  carrier_df   → {len(carrier_df):>4} rows  (per-channel cross-node stats)")
print(f"  node_df      → {len(node_df):>4} rows  (per-node aggregate)")
print(f"  violations   → {len(violations):>4} carriers")
print(f"  warnings_df  → {len(warnings_df):>4} carriers")
print(f"  low_conf     → {len(low_conf):>4} carriers  (confidence < 0.5)")

: 

: 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c71e238e-fa4e-44ac-af7f-d29021a4ac02' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>